In [123]:
import numpy as np 
import pandas as pd

In [124]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

In [125]:
movies.dtypes

budget                    int64
genres                      str
homepage                    str
id                        int64
keywords                    str
original_language           str
original_title              str
overview                    str
popularity              float64
production_companies        str
production_countries        str
release_date                str
revenue                   int64
runtime                 float64
spoken_languages            str
status                      str
tagline                     str
title                       str
vote_average            float64
vote_count                int64
dtype: object

In [126]:
credits.dtypes

movie_id    int64
title         str
cast          str
crew          str
dtype: object

In [127]:
movies = movies.merge(credits,on="title")
movies.dtypes

budget                    int64
genres                      str
homepage                    str
id                        int64
keywords                    str
original_language           str
original_title              str
overview                    str
popularity              float64
production_companies        str
production_countries        str
release_date                str
revenue                   int64
runtime                 float64
spoken_languages            str
status                      str
tagline                     str
title                       str
vote_average            float64
vote_count                int64
movie_id                  int64
cast                        str
crew                        str
dtype: object

In [128]:
movies = movies[['movie_id','title','genres','keywords','overview','cast','crew','release_date','runtime','tagline','budget','revenue']]

In [129]:
movies.isnull().sum()

movie_id          0
title             0
genres            0
keywords          0
overview          3
cast              0
crew              0
release_date      1
runtime           2
tagline         844
budget            0
revenue           0
dtype: int64

In [130]:

movies['overview'] = movies['overview'].fillna('')

text_cols = ['genres', 'keywords', 'cast', 'crew', 'tagline', 'title', 'release_date']
for col in text_cols:
    if col in movies.columns:
        movies[col] = movies[col].fillna('Unknown')

if 'runtime' in movies.columns:
    movies['runtime'] = movies['runtime'].fillna(movies['runtime'].median())
if 'budget' in movies.columns:
    movies['budget'] = movies['budget'].fillna(0)
if 'revenue' in movies.columns:
    movies['revenue'] = movies['revenue'].fillna(0)

In [131]:
movies.isnull().sum()

movie_id        0
title           0
genres          0
keywords        0
overview        0
cast            0
crew            0
release_date    0
runtime         0
tagline         0
budget          0
revenue         0
dtype: int64

In [132]:
movies.duplicated().sum()

np.int64(0)

In [133]:
import ast
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [134]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [135]:
def convert5(obj):
    c = 0
    L = []
    for i in ast.literal_eval(obj):
        if c != 5:
            L.append(i['name'])
            c=+1
        else:
            break
    return L

In [136]:
movies['cast'] = movies['cast'].apply(convert5)

In [137]:
def fetch_dir(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

In [138]:
movies['crew'] = movies['crew'].apply(fetch_dir)

In [139]:
o = movies['overview'].apply(lambda x:x.split())
g = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
k = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
c = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
d = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

movies["tag"] = o+g+k+c+d

In [140]:
movies['tag'] = movies['tag'].apply(lambda x:" ".join(x))

In [141]:
movies['tag'] = movies['tag'].apply(lambda x:x.lower())

In [142]:
new = movies[['movie_id','title','tag']]

In [143]:
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [144]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [145]:
new['tag'] = new['tag'].apply(stem)
new

,movie_id,title,tag
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,206647,Spectre,a cryptic messag from bond’ past send him on a...
3,49026,The Dark Knight Rises,follow the death of district attorney harvey d...
4,49529,John Carter,"john carter is a war-weary, former militari ca..."
...,...,...,...
4804,9367,El Mariachi,el mariachi just want to play hi guitar and ca...
4805,72766,Newlyweds,a newlyw couple' honeymoon is upend by the arr...
4806,231617,"Signed, Sealed, Delivered","""signed, sealed, delivered"" introduc a dedic q..."
4807,126186,Shanghai Calling,when ambiti new york attorney sam is sent to s...


In [146]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english') 

In [147]:
vectors = cv.fit_transform(new['tag']).toarray()
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(4809, 5000))

In [148]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)

In [149]:
print(movies.columns.tolist())

['movie_id', 'title', 'genres', 'keywords', 'overview', 'cast', 'crew', 'release_date', 'runtime', 'tagline', 'budget', 'revenue', 'tag']


In [150]:
import pickle

# Merge tag data with full movie metadata
movie_data = new[['movie_id', 'title', 'tag']].merge(
    movies[['movie_id', 'overview', 'genres', 'tagline', 'budget', 'revenue', 'release_date', 'runtime']],
    on='movie_id',
    how='left'
)

print("Columns:", movie_data.columns.tolist())
print(movie_data.head(2))

pickle.dump(movie_data, open('movie_list.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
print("Done ✓")

Columns: ['movie_id', 'title', 'tag', 'overview', 'genres', 'tagline', 'budget', 'revenue', 'release_date', 'runtime']
   movie_id                                     title  \
0     19995                                    Avatar   
1       285  Pirates of the Caribbean: At World's End   

                                                 tag  \
0  in the 22nd century, a parapleg marin is dispa...   
1  captain barbossa, long believ to be dead, ha c...   

                                            overview  \
0  In the 22nd century, a paraplegic Marine is di...   
1  Captain Barbossa, long believed to be dead, ha...   

                                          genres  \
0  [Action, Adventure, Fantasy, Science Fiction]   
1                   [Adventure, Fantasy, Action]   

                                          tagline     budget     revenue  \
0                     Enter the World of Pandora.  237000000  2787965087   
1  At the end of the world, the adventure begins.  300000000  

In [151]:
import numpy as np 
import pandas as pd

In [152]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

In [153]:
movies.dtypes

budget                    int64
genres                      str
homepage                    str
id                        int64
keywords                    str
original_language           str
original_title              str
overview                    str
popularity              float64
production_companies        str
production_countries        str
release_date                str
revenue                   int64
runtime                 float64
spoken_languages            str
status                      str
tagline                     str
title                       str
vote_average            float64
vote_count                int64
dtype: object

In [154]:
credits.dtypes

movie_id    int64
title         str
cast          str
crew          str
dtype: object

In [155]:
movies = movies.merge(credits,on="title")
movies.dtypes

budget                    int64
genres                      str
homepage                    str
id                        int64
keywords                    str
original_language           str
original_title              str
overview                    str
popularity              float64
production_companies        str
production_countries        str
release_date                str
revenue                   int64
runtime                 float64
spoken_languages            str
status                      str
tagline                     str
title                       str
vote_average            float64
vote_count                int64
movie_id                  int64
cast                        str
crew                        str
dtype: object

In [156]:
movies = movies[['movie_id','title','genres','keywords','overview','cast','crew','release_date','runtime','tagline','budget','revenue']]

In [157]:
movies.isnull().sum()

movie_id          0
title             0
genres            0
keywords          0
overview          3
cast              0
crew              0
release_date      1
runtime           2
tagline         844
budget            0
revenue           0
dtype: int64

In [158]:

movies['overview'] = movies['overview'].fillna('')

text_cols = ['genres', 'keywords', 'cast', 'crew', 'tagline', 'title', 'release_date']
for col in text_cols:
    if col in movies.columns:
        movies[col] = movies[col].fillna('Unknown')

if 'runtime' in movies.columns:
    movies['runtime'] = movies['runtime'].fillna(movies['runtime'].median())
if 'budget' in movies.columns:
    movies['budget'] = movies['budget'].fillna(0)
if 'revenue' in movies.columns:
    movies['revenue'] = movies['revenue'].fillna(0)

In [159]:
movies.isnull().sum()

movie_id        0
title           0
genres          0
keywords        0
overview        0
cast            0
crew            0
release_date    0
runtime         0
tagline         0
budget          0
revenue         0
dtype: int64

In [160]:
movies.duplicated().sum()

np.int64(0)

In [161]:
import ast
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [162]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [163]:
def convert5(obj):
    c = 0
    L = []
    for i in ast.literal_eval(obj):
        if c != 5:
            L.append(i['name'])
            c=+1
        else:
            break
    return L

In [164]:
movies['cast'] = movies['cast'].apply(convert5)

In [165]:
def fetch_dir(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

In [166]:
movies['crew'] = movies['crew'].apply(fetch_dir)

In [167]:
o = movies['overview'].apply(lambda x:x.split())
g = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
k = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
c = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
d = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

movies["tag"] = o+g+k+c+d

In [168]:
movies['tag'] = movies['tag'].apply(lambda x:" ".join(x))

In [169]:
movies['tag'] = movies['tag'].apply(lambda x:x.lower())

In [170]:
new = movies[['movie_id','title','tag']]

In [171]:
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [172]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [173]:
new['tag'] = new['tag'].apply(stem)
new

,movie_id,title,tag
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,206647,Spectre,a cryptic messag from bond’ past send him on a...
3,49026,The Dark Knight Rises,follow the death of district attorney harvey d...
4,49529,John Carter,"john carter is a war-weary, former militari ca..."
...,...,...,...
4804,9367,El Mariachi,el mariachi just want to play hi guitar and ca...
4805,72766,Newlyweds,a newlyw couple' honeymoon is upend by the arr...
4806,231617,"Signed, Sealed, Delivered","""signed, sealed, delivered"" introduc a dedic q..."
4807,126186,Shanghai Calling,when ambiti new york attorney sam is sent to s...


In [174]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english') 

In [175]:
vectors = cv.fit_transform(new['tag']).toarray()
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(4809, 5000))

In [176]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)

In [177]:
import pickle

# Merge tag data with full movie metadata
movie_data = new[['movie_id', 'title', 'tag']].merge(
    movies[['movie_id', 'overview', 'genres', 'tagline', 'budget', 'revenue', 'release_date', 'runtime']],
    on='movie_id',
    how='left'
)

print("Columns:", movie_data.columns.tolist())
print(movie_data.head(2))

pickle.dump(movie_data, open('movie_list.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
print("Done ✓")

Columns: ['movie_id', 'title', 'tag', 'overview', 'genres', 'tagline', 'budget', 'revenue', 'release_date', 'runtime']
   movie_id                                     title  \
0     19995                                    Avatar   
1       285  Pirates of the Caribbean: At World's End   

                                                 tag  \
0  in the 22nd century, a parapleg marin is dispa...   
1  captain barbossa, long believ to be dead, ha c...   

                                            overview  \
0  In the 22nd century, a paraplegic Marine is di...   
1  Captain Barbossa, long believed to be dead, ha...   

                                          genres  \
0  [Action, Adventure, Fantasy, Science Fiction]   
1                   [Adventure, Fantasy, Action]   

                                          tagline     budget     revenue  \
0                     Enter the World of Pandora.  237000000  2787965087   
1  At the end of the world, the adventure begins.  300000000  

In [178]:
pip install rapidfuzz

Note: you may need to restart the kernel to use updated packages.


In [179]:
from rapidfuzz import process

def recommend(movie):
    movie_titles = movies['title'].tolist()

    match, score, _ = process.extractOne(movie, movie_titles)

    if score < 50:
        print("Movie not found")
        return

    print(f"\nRecommendations for '{match}':\n")

    index = movies[movies['title'] == match].index[0]

    distances = sorted(
        list(enumerate(similarity[index])),
        key=lambda x: x[1],
        reverse=True
    )

    for i in distances[1:6]:
        print("•", new.iloc[i[0]]['title'])

In [180]:
recommend("avengers")


Recommendations for 'The Avengers':

• Avengers: Age of Ultron
• Iron Man 3
• Iron Man
• Captain America: Civil War
• Iron Man 2


In [181]:
pip install rapidfuzz

Note: you may need to restart the kernel to use updated packages.


In [182]:
from rapidfuzz import process

def recommend(movie):
    movie_titles = movies['title'].tolist()

    match, score, _ = process.extractOne(movie, movie_titles)

    if score < 50:
        print("Movie not found")
        return

    print(f"\nRecommendations for '{match}':\n")

    index = movies[movies['title'] == match].index[0]

    distances = sorted(
        list(enumerate(similarity[index])),
        key=lambda x: x[1],
        reverse=True
    )

    for i in distances[1:6]:
        print("•", new.iloc[i[0]]['title'])

In [ ]:
recommend("avengers")


Recommendations for 'The Avengers':

• Avengers: Age of Ultron
• Iron Man 3
• Iron Man
• Captain America: Civil War
• Iron Man 2


: 